# Descriptive research analysis

This notebook reads `data/processed/abuse_flags_latest.csv` and reports descriptive feed counts and domain-age distributions. Heuristic labels and registrant categories are candidates, not verified facts. URLhaus is community-sourced URL intelligence, not ground truth for registrar actions or compromise. OpenPhish has no per-URL event timestamp; use `observed_at` as collection time. These summaries are not accuracy or false-positive rates without independently reviewed labels.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / 'data' / 'processed').exists():
    project_root = project_root.parent
data_path = project_root / 'data' / 'processed' / 'abuse_flags_latest.csv'
if not data_path.exists():
    raise FileNotFoundError(f'{data_path} not found. Run `docker compose run --rm pipeline` first.')
df = pd.read_csv(data_path, parse_dates=['flagged_at', 'observed_at'])
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
print(f'{df.flag_id.nunique():,} unique feed URLs across {df.source_feed.nunique()} feeds')
display(df.groupby(['source_feed', 'classification_label'], dropna=False).agg(unique_feed_urls=('flag_id', 'nunique'), sightings=('flagged_url', 'size')).reset_index())

In [ ]:
age = pd.to_numeric(df['domain_age_at_flag_days'], errors='coerce').dropna()
if age.empty:
    print('No RDAP-derived domain ages are available in this export.')
else:
    ax = age.plot.hist(bins=30, figsize=(9, 4), edgecolor='white')
    ax.set(title='Observed domain age at flag time', xlabel='Age (days)', ylabel='Observations')
    plt.tight_layout()
    plt.show()

In [ ]:
summary = (df.groupby(['classification_label', 'registrant_category'], dropna=False)
             .agg(unique_feed_urls=('flag_id', 'nunique'),
                  mean_domain_age_days=('domain_age_at_flag_days', 'mean'),
                  reviewed_rows=('human_review_status', lambda s: s.fillna('not_reviewed').ne('not_reviewed').sum()))
             .reset_index()
             .sort_values('unique_feed_urls', ascending=False))
summary